In [5]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage

In [15]:
#Specialised reducer
from langgraph.graph.message import add_messages

In [16]:
from typing import Literal, TypedDict, Optional, Annotated

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [43]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0.7, max_output_tokens=512)

In [44]:
def chat_with_ai(state: ChatState):
    messages = state['messages']
    prompt_template = PromptTemplate(
        input_variables=["messages"],
        template="You are a helpful assistant. Continue the conversation based on the following messages. Reply in short-oneline if possi:\n\n{messages}"
    )
    chain = prompt_template | llm | StrOutputParser()
    response = chain.invoke({"messages" : messages})
    return  {"messages": [AIMessage(content=response)]}
    

In [45]:
graph = StateGraph(ChatState)

In [46]:
graph.add_node('chat_node' , chat_with_ai)

#add_edges
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

In [47]:
workflow = graph.compile()

In [48]:
input_state = {
    "messages" : [
        SystemMessage(content="You are a Pre-historic animals expert. Be precise and concise in your answers."),
        HumanMessage(content="What is the largest dinosaur that ever lived?")
    ]
}

In [49]:
final_state = workflow.invoke(input_state)

In [50]:
for message in final_state['messages']:
    print(f"{message.type}: {message.content}")

system: You are a Pre-historic animals expert. Be precise and concise in your answers.
human: What is the largest dinosaur that ever lived?
ai: The largest known dinosaur is *Argentinosaurus*, weighing around 70-100 metric tons.
